# Pixelization Recipe — student walkthrough

## Learning to Autolens / Module 05

---

**What this notebook is.** A focused recipe-card for pixelized source reconstructions. Sister notebook to the main `05_pixelized_sources.ipynb` — that one explains the *theory* (Voronoi mesh, regularization, inversions); this one is the *operational* recipe you'd reach for when starting a new fit on a new dataset.

**When to use pixelized sources** (not just parametric):
1. The source has **complex morphology** that a Sersic can't describe — multiple knots, asymmetric arcs, irregular galaxy.
2. You need **per-pixel residual maps** in the source plane to look for substructure.
3. Your parametric fit shows **arc-shaped residuals** — the model is missing source structure that a flexible mesh can capture.

**When NOT to use pixelized sources:**
1. The source IS a Sersic / SersicCore — pixelization is overkill and slower.
2. Your data is single-band imaging without good S/N — the inversion will over-fit noise into the source plane.
3. The lens model is wildly wrong — pixelization will absorb mass-model errors into the source, confusing the diagnostic.

**The recipe in 6 steps.** Each cell below is one step. The full converged result for the Mod 05 dataset is in `results/search2_pixelized_source/`.

---

## Step 1 — load + mask the data, recover the parametric baseline

**Why first.** The pixelized inversion needs a converged mass model as its starting point. Run the parametric fit first; pixelization passes the lens-mass posterior in as priors.

Reference: `05_pixelized_sources.ipynb` §1-2 covers this — Module 05's existing parametric search (`search1_parametric_source`) is the converged mass baseline we'll feed in.

In [ ]:
import os
os.environ.setdefault("PYAUTOFIT_TEST_MODE", "1")
from pathlib import Path
import json
import autofit as af
import autolens as al
import autolens.plot as aplt
from IPython.display import Image, Markdown, display
%matplotlib inline

# Module 05 dataset (the canonical Iso lens with complex source)
dataset_path = Path("../../autolens_workspace_latest/dataset/imaging/simple")
if not dataset_path.exists():
    dataset_path = Path("../../autolens_workspace_original/dataset/imaging/no_lens_light/mass_sie__source_sersic")
print(f"Dataset: {dataset_path}")

# Show the converged parametric baseline
param_summary = json.loads(
    Path("results/search1_parametric_source/summary.json").read_text()
)
display(Markdown(
    f"**Step 1 baseline (parametric SersicCore source):** "
    f"χ²/N = {param_summary['chi_squared_per_pixel']:.3f}, "
    f"max\\|res\\| = {param_summary['max_abs_normalized_residual']:.2f}σ, "
    f"log_Z = {param_summary['log_evidence']:.2f}"
))

## Step 2 — choose your mesh + regularization

The two key choices for a pixelized source:

| Choice | What it does | When to use |
|---|---|---|
| **Mesh: `Rectangular` (M×M)** | Fixed M×M grid of source-plane pixels | Simple, deterministic, fast; first cut |
| **Mesh: `Delaunay`** | Adaptive Voronoi triangulation around bright pixels | Better for irregular sources |
| **Mesh: `Hilbert` / `RectangularAdaptDensity`** | Mesh density follows source brightness | Best quality, slowest, requires `adapt_image` |
| **Reg: `Constant`** | Single λ, uniform across mesh | First cut |
| **Reg: `AdaptBrightness`** | λ varies — strong on faint regions, weak on bright | Modern default |
| **Reg: `AdaptSplit`** | Adaptive + signal-split (2026.4 only) | Best quality |

**Recommendation for first time:** `RectangularAdaptDensity(shape=(30, 30))` mesh + `AdaptBrightness` regularization. It's the autolens 2026.4 default that the SLaM `source_pix_1` pipeline uses.

In [ ]:
# Construct the pixelization model
pixelization = af.Model(
    al.Pixelization,
    image_mesh=af.Model(al.image_mesh.Hilbert),
    mesh=af.Model(al.mesh.Delaunay),
    regularization=af.Model(al.reg.AdaptSplit),
)
print(f"Pixelization free params: {pixelization.prior_count}")
print(f"  (image_mesh + mesh + regularization combined)")

## Step 3 — pass the mass posterior from Step 1 as priors

**This is the critical step.** A pixelized source can absorb mass-model errors. To prevent that, you fix or tightly constrain the mass parameters at the parametric posterior. PyAutoFit's `result.model.galaxies.lens.mass` does exactly that — passes Gaussian priors centred on the parametric posterior medians.

*In production* you load the parametric `result_1` object directly. *In this recipe* we'll show the API; the actual chained search is in `05_pixelized_sources.ipynb`.

In [ ]:
# Mass priors are passed via result.model.galaxies.lens.mass when chaining searches.
# Here we show the model structure with placeholder priors (see 05_pixelized_sources.ipynb
# for the actual prior-passing pattern).
lens_mass = af.Model(al.mp.Isothermal)
lens_mass.einstein_radius = af.GaussianPrior(mean=1.6, sigma=0.05)  # tight on parametric MAP
lens_mass.centre.centre_0 = af.GaussianPrior(mean=0.0, sigma=0.02)
lens_mass.centre.centre_1 = af.GaussianPrior(mean=0.0, sigma=0.02)
lens_mass.ell_comps.ell_comps_0 = af.GaussianPrior(mean=0.0, sigma=0.05)
lens_mass.ell_comps.ell_comps_1 = af.GaussianPrior(mean=0.0, sigma=0.05)

lens = af.Model(al.Galaxy, redshift=0.5, mass=lens_mass)

# Source: pixelization, no parametric component
source = af.Model(al.Galaxy, redshift=1.0, pixelization=pixelization)

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))
print(f"Pixelized model total free params: {model.prior_count}")
print("  (lens mass tightly constrained from parametric Step 1, pixelization knobs free)")

## Step 4 — `AnalysisImaging` with `adapt_image`

Adaptive pixelizations need an `adapt_image` to know which source-plane regions are bright (so the mesh can refine there). Standard recipe: pass the parametric Step 1 max-likelihood reconstructed image as the adapt image.

**Without an adapt image,** the mesh density is uniform — falls back to a simple grid. Adaptive features turn off.

In [ ]:
# In practice you load the converged parametric result and extract its adapt image:
#   adapt_images = al.AdaptImages(
#       galaxy_image_dict={
#           "source": result_1.adapt_image_max_log_likelihood,
#       }
#   )
#   analysis = al.AnalysisImaging(dataset=dataset, adapt_images=adapt_images)
#
# Without an adapt image, AdaptBrightness regularization falls back to Constant.
print("adapt_image is the parametric Step 1 max-likelihood source reconstruction.")
print("Without one: lose adaptive density + adaptive regularization.")

## Step 5 — search settings + position likelihood

**Pixelized fits hate exploration.** The mesh + regularization parameter space has lots of degenerate minima. You want:
1. **Tight mass priors** from Step 1 (already done).
2. **`positions_likelihood_list=[al.PositionsLH(positions=positions, threshold=0.1)]`** — image positions tighten the mass-model search to the right basin.
3. **`n_live ~ 200`** — pixelized inversions are expensive; don't burn cycles on huge n_live.
4. **`use_jax=False`** — the JAX path can be slower for inversions in our autolens version (CPU multiprocessing wins for 9k-pixel imaging).

Wall-time expectation: **~2-4× the parametric search cost** for a single pixelized search. The full SLaM `source_pix_1` + `source_pix_2` is 2 sequential searches because the second uses a higher-quality mesh seeded from the first.

## Step 6 — read the residuals

Pixelized fit auditing has its own panel walk. The committed `search2_pixelized_source/fit_subplot.png` is the reference.

In [ ]:
fp = Path("results/search2_pixelized_source/fit_subplot.png")
summary = json.loads(Path("results/search2_pixelized_source/summary.json").read_text())
display(Markdown(
    f"**Step 6 result (pixelized source):**\n"
    f"- log_Z = {summary['log_evidence']:.2f}\n"
    f"- χ²/N = {summary['chi_squared_per_pixel']:.3f}\n"
    f"- max\\|res\\| = {summary['max_abs_normalized_residual']:.2f}σ\n"
))
if fp.exists():
    display(Image(filename=str(fp)))

**Audit checklist for pixelized fits:**

1. **Image-plane Normalized Residual Map** — clean white-noise everywhere, no arc structure.
2. **Source Plane (Zoomed)** — recognizable galaxy structure, not a single bright pixel (mesh collapse) or a diffuse blob (over-regularized).
3. **Caustic** — look for a clean diamond / butterfly shape. Pathological caustics (extended, knotted) signal a mass-model problem the pixelization absorbed.
4. **Log evidence** — should beat the parametric baseline by ≥ a few hundred (the inversion + N source pixels add a lot of model freedom; the data should reward it).

**Common failure modes:**
- **Single-pixel collapse** — source plane shows one bright pixel. Cause: regularization too weak, mesh too fine. Fix: increase λ floor.
- **Diffuse blob** — source has no structure, just a smooth blob. Cause: regularization too strong, OR mass model is so wrong the inversion couldn't recover the source. Fix: re-check Step 1 parametric fit, then loosen λ.
- **Image-plane arc residuals** — the mass model is wrong. Pixelization can absorb some but not all errors. Fix: go back to the parametric search with better priors / more parameters.

---

## See also

- [`05_pixelized_sources.ipynb`](05_pixelized_sources.ipynb) — the main module notebook with full theoretical treatment
- [`Modules/09_MGE_Linear_Light_Profiles/05_mge_recipe.ipynb`](../09_MGE_Linear_Light_Profiles/05_mge_recipe.ipynb) — sister recipe for MGE light
- [`Examples/compound_lens_zoo/03_slam_recipe.ipynb`](../../Examples/compound_lens_zoo/03_slam_recipe.ipynb) — sister recipe for SLaM staging
- `autolens_workspace_latest/scripts/imaging/features/pixelization/modeling.py` — canonical PyAutoLens pixelization example
- `autolens_workspace_latest/scripts/imaging/features/pixelization/adaptive.py` — adaptive mesh deep-dive
- Suyu+ 2006 (Bayesian source reconstruction), Vegetti+ 2009 (adaptive grid pixelization)